In [2]:
#DECODELABS INTERNSHIP - DATA SCIENCE PROJECT 3 - Amashi Fernando
#Unsupervised Learning (Customer Segmentation)
#Dataset: marketing campaign

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples

!pip install kneed
from kneed import KneeLocator

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
OUTPUT_DIR = "outputs"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [5]:
# Load data

# The original Kaggle file is TAB-separated, not comma-separated.
df = pd.read_csv("Dataset-marketing_campaign.csv", sep="\t")
print(f"Raw shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}\n")
print(df.head())

Raw shape: 2240 rows x 29 columns
Columns: ['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response']

     ID  Year_Birth   Education Marital_Status   Income  Kidhome  Teenhome  \
0  5524        1957  Graduation         Single  58138.0        0         0   
1  2174        1954  Graduation         Single  46344.0        1         1   
2  4141        1965  Graduation       Together  71613.0        0         0   
3  6182        1984  Graduation       Together  26646.0        1         0   
4  5324        1981         PhD        Married  58293.0        1         0   

  Dt_Customer  Recency  MntWines  ... 

In [6]:
# Data Wrangling

# --- Missing values ---
print("Missing values per column (only columns with > 0 shown):")
missing = df.isnull().sum()
print(missing[missing > 0])

# Income has a small number of missing values -> impute with median (robust to outliers)
df["Income"] = df["Income"].fillna(df["Income"].median())

# --- Duplicates ---
n_dupes = df.duplicated().sum()
print(f"\nDuplicate rows: {n_dupes}")
df = df.drop_duplicates()

# --- Fix rare/typo category labels in Marital_Status ---
df["Marital_Status"] = df["Marital_Status"].replace(
    {"Alone": "Single", "Absurd": "Single", "YOLO": "Single"}
)

# --- Remove obvious data-entry outliers ---
# A small number of rows have implausible birth years (e.g. 1893, 1899, 1900)
# and one or two incomes in the millions. We cap these rather than silently
# dropping large chunks of data.
df = df[df["Year_Birth"] > 1930]
df = df[df["Income"] < 200000]
print(f"\nShape after removing extreme outliers: {df.shape}")

# --- Z_CostContact and Z_Revenue are constant columns (no variance) , dropped ---
df = df.drop(columns=["Z_CostContact", "Z_Revenue"], errors="ignore")



Missing values per column (only columns with > 0 shown):
Income    24
dtype: int64

Duplicate rows: 0

Shape after removing extreme outliers: (2236, 29)


In [7]:
# Feature Engineering

# --- Age (relative to dataset's reference year, 2014, when it was collected) ---
df["Age"] = 2014 - df["Year_Birth"]

# --- Customer tenure in days, from enrollment date to most recent date in data ---
df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
most_recent_date = df["Dt_Customer"].max()
df["Customer_Tenure_Days"] = (most_recent_date - df["Dt_Customer"]).dt.days

# --- Total spend across all product categories ---
spend_cols = ["MntWines", "MntFruits", "MntMeatProducts",
              "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
df["Total_Spend"] = df[spend_cols].sum(axis=1)

# --- Total purchases across all channels ---
purchase_cols = ["NumDealsPurchases", "NumWebPurchases",
                  "NumCatalogPurchases", "NumStorePurchases"]
df["Total_Purchases"] = df[purchase_cols].sum(axis=1)

# --- Total accepted marketing campaigns (loyalty/engagement signal) ---
campaign_cols = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3",
                  "AcceptedCmp4", "AcceptedCmp5", "Response"]
df["Total_Campaigns_Accepted"] = df[campaign_cols].sum(axis=1)

# --- Total dependents at home ---
df["Total_Children"] = df["Kidhome"] + df["Teenhome"]

# --- Average spend per purchase (avoids divide-by-zero) ---
df["Avg_Spend_Per_Purchase"] = df["Total_Spend"] / df["Total_Purchases"].replace(0, 1)

# --- Encode Education ordinally (it has a natural order) ---
education_order = {"Basic": 0, "2n Cycle": 1, "Graduation": 2, "Master": 3, "PhD": 4}
df["Education_Encoded"] = df["Education"].map(education_order)

# --- One-hot encode Marital_Status (no natural order) ---
df = pd.get_dummies(df, columns=["Marital_Status"], prefix="Marital", drop_first=False)

print(f"Shape after feature engineering: {df.shape}")
print(f"New engineered features: Age, Customer_Tenure_Days, Total_Spend, "
      f"Total_Purchases, Total_Campaigns_Accepted, Total_Children, "
      f"Avg_Spend_Per_Purchase, Education_Encoded, Marital_* dummies\n")



Shape after feature engineering: (2236, 39)
New engineered features: Age, Customer_Tenure_Days, Total_Spend, Total_Purchases, Total_Campaigns_Accepted, Total_Children, Avg_Spend_Per_Purchase, Education_Encoded, Marital_* dummies



In [8]:
# Feature selection for clustering

# Drop identifier and raw/redundant columns that shouldn't enter distance calcs
drop_cols = ["ID", "Year_Birth", "Dt_Customer", "Education"]
cluster_df = df.drop(columns=drop_cols, errors="ignore")

# Keep only numeric columns for clustering
cluster_df = cluster_df.select_dtypes(include=[np.number])

print(f"Final feature count going into PCA: {cluster_df.shape[1]} columns")
print(f"Features: {list(cluster_df.columns)}\n")

feature_names = cluster_df.columns.tolist()


Final feature count going into PCA: 30 columns
Features: ['Income', 'Kidhome', 'Teenhome', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Response', 'Age', 'Customer_Tenure_Days', 'Total_Spend', 'Total_Purchases', 'Total_Campaigns_Accepted', 'Total_Children', 'Avg_Spend_Per_Purchase', 'Education_Encoded']



In [9]:
# Scaling (Standard Scaler)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(cluster_df)
print(f"Scaled matrix shape: {X_scaled.shape}")
print(f"Mean ~0, Std ~1 check -> mean: {X_scaled.mean():.4f}, std: {X_scaled.std():.4f}\n")



Scaled matrix shape: (2236, 30)
Mean ~0, Std ~1 check -> mean: 0.0000, std: 1.0000



In [10]:
# Compress (Dimensionality Reduction)

# First, fit PCA with all components to inspect the explained-variance curve
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

cum_var = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = np.argmax(cum_var >= 0.95) + 1
print(f"Components needed to retain 95% variance: {n_components_95}")

# Plot cumulative explained variance (the "95% Rule" chart from the brief)
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(cum_var) + 1), cum_var, marker="o", color="#2c3e50")
plt.axhline(0.95, color="#e67e22", linestyle="--", label="95% threshold")
plt.axvline(n_components_95, color="#e67e22", linestyle=":")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA — Cumulative Explained Variance")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/01_pca_explained_variance.png")
plt.close()
print(f"Saved: {OUTPUT_DIR}/01_pca_explained_variance.png")

# For clustering + visualization, reduce to 3 components.

N_PCA_COMPONENTS = 3
pca = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print(f"\nUsing {N_PCA_COMPONENTS} components for clustering.")
print(f"Variance explained by these {N_PCA_COMPONENTS} components: "
      f"{pca.explained_variance_ratio_.sum():.2%}")
for i, var in enumerate(pca.explained_variance_ratio_, 1):
    print(f"  PC{i}: {var:.2%}")

# Inspect which original features load most heavily onto each PC
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f"PC{i+1}" for i in range(N_PCA_COMPONENTS)],
    index=feature_names
)
print("\nTop 5 contributing features per component:")
for col in loadings.columns:
    top5 = loadings[col].abs().sort_values(ascending=False).head(5)
    print(f"\n{col}:")
    for feat, val in top5.items():
        print(f"  {feat}: {loadings.loc[feat, col]:+.3f}")
print()



Components needed to retain 95% variance: 21
Saved: outputs/01_pca_explained_variance.png

Using 3 components for clustering.
Variance explained by these 3 components: 48.22%
  PC1: 30.53%
  PC2: 9.12%
  PC3: 8.56%

Top 5 contributing features per component:

PC1:
  Total_Spend: +0.316
  Income: +0.275
  MntWines: +0.270
  NumCatalogPurchases: +0.269
  MntMeatProducts: +0.267

PC2:
  Teenhome: +0.399
  NumDealsPurchases: +0.358
  Total_Campaigns_Accepted: -0.320
  Total_Purchases: +0.299
  NumWebPurchases: +0.271

PC3:
  Total_Campaigns_Accepted: +0.399
  Response: +0.308
  AcceptedCmp4: +0.305
  NumDealsPurchases: +0.266
  NumWebVisitsMonth: +0.258



In [11]:
# Determine optimal K, then fit K-Means

K_RANGE = range(2, 11)
wcss = []
sil_scores = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_pca)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_pca, labels))

# --- Gatekeeper 1: Elbow Method (automated via Kneedle algorithm) ---
knee = KneeLocator(list(K_RANGE), wcss, curve="convex", direction="decreasing")
elbow_k = knee.elbow
print(f"Elbow Method suggests K = {elbow_k}")

# --- Gatekeeper 2: Silhouette Score ---
best_sil_k = list(K_RANGE)[int(np.argmax(sil_scores))]
print(f"Best Silhouette Score is at K = {best_sil_k} "
      f"(score = {max(sil_scores):.4f})")

for k, w, s in zip(K_RANGE, wcss, sil_scores):
    marker = ""
    if k == elbow_k:
        marker += " <- elbow"
    if k == best_sil_k:
        marker += " <- best silhouette"
    print(f"  K={k}: WCSS={w:,.1f}, Silhouette={s:.4f}{marker}")

# Plot both diagnostics side by side
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(list(K_RANGE), wcss, marker="o", color="#2c3e50")
axes[0].axvline(elbow_k, color="#c0392b", linestyle="--", label=f"Elbow (K={elbow_k})")
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("WCSS (Inertia)")
axes[0].set_title("Elbow Method")
axes[0].legend()

axes[1].plot(list(K_RANGE), sil_scores, marker="o", color="#16a085")
axes[1].axvline(best_sil_k, color="#c0392b", linestyle="--",
                 label=f"Best K={best_sil_k}")
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score")
axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/02_elbow_and_silhouette.png")
plt.close()
print(f"\nSaved: {OUTPUT_DIR}/02_elbow_and_silhouette.png")

# Final K choice:
# Elbow and Silhouette can disagree (a very common, expected real-world
# finding -- not a bug). Here, K=2 maximizes silhouette but only separates
# customers by income into two broad blobs, which is too coarse to be a
# useful business segmentation. K=4 (the elbow point) gives more actionable,
# distinct personas while still scoring a healthy silhouette. We therefore
# choose K on elbow + interpretability rather than pure silhouette score.
# Document this trade-off explicitly in your report -- it's good practice,
# not a flaw, to override a metric when business interpretability suffers.
FINAL_K = elbow_k
print(f"\n>>> FINAL CHOSEN K = {FINAL_K} "
      f"(elbow={elbow_k}, best-silhouette={best_sil_k}) <<<\n")


Elbow Method suggests K = 4
Best Silhouette Score is at K = 2 (score = 0.4655)
  K=2: WCSS=16,445.8, Silhouette=0.4655 <- best silhouette
  K=3: WCSS=11,921.3, Silhouette=0.4184
  K=4: WCSS=8,709.7, Silhouette=0.4317 <- elbow
  K=5: WCSS=7,621.2, Silhouette=0.3591
  K=6: WCSS=6,793.0, Silhouette=0.3308
  K=7: WCSS=6,053.2, Silhouette=0.3345
  K=8: WCSS=5,437.1, Silhouette=0.3228
  K=9: WCSS=4,922.7, Silhouette=0.3321
  K=10: WCSS=4,477.4, Silhouette=0.3484

Saved: outputs/02_elbow_and_silhouette.png

>>> FINAL CHOSEN K = 4 (elbow=4, best-silhouette=2) <<<



In [12]:
# Fit final K-means model

kmeans_final = KMeans(n_clusters=FINAL_K, init="k-means++", n_init=10,
                       random_state=RANDOM_STATE)
cluster_labels = kmeans_final.fit_predict(X_pca)

df_result = df.copy()
df_result["Cluster"] = cluster_labels
cluster_df["Cluster"] = cluster_labels

final_silhouette = silhouette_score(X_pca, cluster_labels)
print(f"Final Silhouette Score: {final_silhouette:.4f}")
print(f"Cluster sizes:\n{df_result['Cluster'].value_counts().sort_index()}\n")


Final Silhouette Score: 0.4317
Cluster sizes:
Cluster
0    1030
1     582
2     453
3     171
Name: count, dtype: int64



In [13]:
# Visualizing Clusters

# --- 2D scatter (PC1 vs PC2) ---
plt.figure(figsize=(8, 6))
palette = sns.color_palette("Set2", FINAL_K)
for c in range(FINAL_K):
    mask = cluster_labels == c
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1],
                label=f"Cluster {c}", alpha=0.6, color=palette[c])
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.title(f"Customer Segments in PCA Space (K={FINAL_K})")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/03_clusters_2d.png")
plt.close()
print(f"Saved: {OUTPUT_DIR}/03_clusters_2d.png")

# --- 3D scatter (PC1 vs PC2 vs PC3) ---
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
for c in range(FINAL_K):
    mask = cluster_labels == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
               label=f"Cluster {c}", alpha=0.6, color=palette[c])
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title(f"Customer Segments in 3D PCA Space (K={FINAL_K})")
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/04_clusters_3d.png")
plt.close()
print(f"Saved: {OUTPUT_DIR}/04_clusters_3d.png")

# --- Silhouette plot (per-sample diagnostic, common in professional reports) ---
sample_silhouette_values = silhouette_samples(X_pca, cluster_labels)
plt.figure(figsize=(8, 6))
y_lower = 10
for c in range(FINAL_K):
    cluster_sil_vals = sample_silhouette_values[cluster_labels == c]
    cluster_sil_vals.sort()
    size_c = cluster_sil_vals.shape[0]
    y_upper = y_lower + size_c
    plt.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil_vals,
                       facecolor=palette[c], edgecolor=palette[c], alpha=0.7)
    plt.text(-0.05, y_lower + 0.5 * size_c, str(c))
    y_lower = y_upper + 10
plt.axvline(final_silhouette, color="red", linestyle="--",
            label=f"Average score: {final_silhouette:.3f}")
plt.xlabel("Silhouette Coefficient")
plt.ylabel("Cluster")
plt.title("Silhouette Plot per Cluster")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/05_silhouette_plot.png")
plt.close()
print(f"Saved: {OUTPUT_DIR}/05_silhouette_plot.png\n")


Saved: outputs/03_clusters_2d.png
Saved: outputs/04_clusters_3d.png
Saved: outputs/05_silhouette_plot.png



In [14]:
# Reverse-engineer centroids back to original scale

# Centroids currently live in PCA space. Inverse-transform PCA, then
# inverse-transform the scaler, to recover values in original units.
centroids_pca_space = kmeans_final.cluster_centers_
centroids_scaled_space = pca.inverse_transform(centroids_pca_space)
centroids_original_space = scaler.inverse_transform(centroids_scaled_space)

centroids_df = pd.DataFrame(centroids_original_space, columns=feature_names)
centroids_df.index.name = "Cluster"
print("Cluster centroids (original units):")
key_cols = ["Age", "Income", "Total_Spend", "Total_Purchases",
            "Total_Campaigns_Accepted", "Recency", "Total_Children"]
key_cols = [c for c in key_cols if c in centroids_df.columns]
print(centroids_df[key_cols].round(1))
print()

# Also compute simple groupby means directly from the original data as a
# sanity cross-check against the reverse-engineered centroids above.
profile_cols = key_cols + ["Education_Encoded"]
profile_cols = [c for c in profile_cols if c in df_result.columns]
cluster_profile = df_result.groupby("Cluster")[profile_cols].mean().round(1)
print("Cross-check — actual per-cluster means from raw assigned data:")
print(cluster_profile)
print()

Cluster centroids (original units):
          Age   Income  Total_Spend  Total_Purchases  \
Cluster                                                
0        41.9  35253.8         86.2              8.0   
1        50.8  57488.5        738.9             20.5   
2        45.8  72931.3       1243.1             20.8   
3        43.1  78249.8       1600.3             21.3   

         Total_Campaigns_Accepted  Recency  Total_Children  
Cluster                                                     
0                             0.2     48.3             1.2  
1                             0.3     50.4             1.4  
2                             0.3     52.5             0.2  
3                             2.9     40.4             0.1  

Cross-check — actual per-cluster means from raw assigned data:
          Age   Income  Total_Spend  Total_Purchases  \
Cluster                                                
0        42.4  35085.0         96.0              7.8   
1        49.4  56945.2       

In [15]:
# Business Persona Summary

persona_summary = df_result.groupby("Cluster").agg(
    Count=("Cluster", "size"),
    Avg_Age=("Age", "mean"),
    Avg_Income=("Income", "mean"),
    Avg_Total_Spend=("Total_Spend", "mean"),
    Avg_Purchases=("Total_Purchases", "mean"),
    Avg_Campaigns_Accepted=("Total_Campaigns_Accepted", "mean"),
    Avg_Children=("Total_Children", "mean"),
    Avg_Web_Visits=("NumWebVisitsMonth", "mean"),
).round(1)
persona_summary["Pct_of_Base"] = (
    persona_summary["Count"] / persona_summary["Count"].sum() * 100
).round(1)

print(persona_summary)

# Save everything needed for the report
df_result.to_csv(f"{OUTPUT_DIR}/clustered_customers.csv", index=False)
persona_summary.to_csv(f"{OUTPUT_DIR}/persona_summary.csv")
centroids_df.to_csv(f"{OUTPUT_DIR}/cluster_centroids_original_scale.csv")


         Count  Avg_Age  Avg_Income  Avg_Total_Spend  Avg_Purchases  \
Cluster                                                               
0         1030     42.4     35085.0             96.0            7.8   
1          582     49.4     56945.2            707.2           21.0   
2          453     46.2     73642.1           1266.0           20.8   
3          171     44.0     79102.4           1584.8           20.9   

         Avg_Campaigns_Accepted  Avg_Children  Avg_Web_Visits  Pct_of_Base  
Cluster                                                                     
0                           0.2           1.2             6.4         46.1  
1                           0.3           1.2             5.9         26.0  
2                           0.3           0.2             2.8         20.3  
3                           2.9           0.2             3.5          7.6  



* Segmented 2,236 customers (after cleaning 24 missing incomes, removing
implausible ages/incomes, and dropping zero-variance columns) using a
Scale → PCA → K-Means pipeline on 30 engineered features.
* Full PCA showed 21 components were needed to retain 95% variance, confirming the dataset
was genuinely high-dimensional; 3 components were used for clustering and visualization, capturing 48.2% of variance and loading mainly on spend/income (PC1), household composition and deal-seeking (PC2), and campaign responsiveness (PC3).
* The Elbow Method and Silhouette Score disagreed
(K=4 vs K=2) — K=2 scored higher (0.4655) but only split customers by income into two broad groups, so K=4 was chosen instead (silhouette
0.4317) for more actionable personas.
* Reverse-engineering the centroids
out of PCA/scaled space back to original units produced four distinct, business-interpretable segments ranging from a large low-spend group
(46% of customers, 35k income, 96 spend) to a small high-value, highly campaign-responsive group (7.6% of customers, 79k income,
1,585 spend, 2.9 campaigns accepted on average).

#### Amashi Fernando
Data Science Intern

DecodeLabs Internship (June 10 - July 10 2026)